# 18. Triton Block Model | Triton Block 模型

**难度：** Medium | **环境：** GPU optional | **标签：** `算子编程`, `Triton`, `Block Model` | **目标人群：** 编译与系统入门者

![Triton Program、Tile 与 Mask 的映射](../docs/public/01_Hardware_Math_and_Systems/18_triton_program_tile_mask.svg)

---

## 本节导读

Triton 使用 Program 和 Tile 把张量空间分块。张量尺寸不能被块大小整除时，最后一个 Program 会产生无效 lane，需要通过 Mask 保护边界访问。

本节从一维覆盖、二维 Tile 布局和边界 Mask 建立空间映射。完成后，你应能计算 Program 数、边界填充和有效索引；这些能力直接连接算子优化路线与 Part 03 的 Triton Kernel。

**关键词：** `program`, `block`, `tile`

---

## 前置阅读

**导语：** 先把 CUDA 执行模型、warp / block 组织和编程模型层级接上，再看 Triton 的 block 视角，会更容易把 program 和张量空间对应起来。

- [08. Programming Models and CUDA/Triton | 编程模型演进](./08_Programming_Models_CUDA_Triton.ipynb)
- [15. CUDA Execution Model | CUDA 执行模型](./15_CUDA_Execution_Model.ipynb)
- [16. Warp Block SharedMemory Basics | Warp、Block 与 Shared Memory 基础](./16_Warp_Block_SharedMemory_Basics.ipynb)


## Q1：Triton 里的 program、block 和 tile 分别是什么？


Triton 的核心不是“写函数”，而是“把张量切成块，然后让每个 program 负责其中一块”。

- **program**：Triton 运行时分配的执行单元，可以理解成一个负责某个块的工作实例。
- **block**：一个 program 负责处理的那部分数据，也就是局部计算单元。
- **tile**：数据切分后的具体块形状，通常是 program 需要加载和处理的张量片段。

三者关系很像“任务实例 - 负责区域 - 切块形状”。如果不先把这层关系想清楚，后面看到 `program_id`、`tl.arange` 和 `mask` 时就很容易只记语法，不知道它们在空间上对应什么。

```mermaid
flowchart TD
    P[program] --> B[block]
    B --> T[tile]
    P --> PID[program_id]
    T --> LOAD[load / compute]
    B --- SPACE[负责一块数据区域]
```


### Q1 机制计算器：Program 如何覆盖一维张量

把 program、block 和 tile 先对上号，再看 kernel 会顺很多。

In [ ]:
def program_coverage(num_elements, block_size):
    """计算 Program 数、总 lane 数和最后一个 Program 的有效元素。"""
    if num_elements <= 0 or block_size <= 0:
        raise ValueError('num_elements and block_size must be positive')
    programs = (num_elements + block_size - 1) // block_size
    launched_lanes = programs * block_size
    last_program_valid = num_elements - (programs - 1) * block_size
    return {
        'programs': programs,
        'launched_lanes': launched_lanes,
        'last_program_valid': last_program_valid,
        'masked_lanes': launched_lanes - num_elements,
    }

for block_size in [64, 128, 256]:
    print(block_size, '->', program_coverage(1000, block_size))
assert program_coverage(1000, 256)['masked_lanes'] == 24


## Q2：为什么 Triton 要强调“先布局，再写细节”？


Triton 的写法先强调数据怎么切块、怎么映射、怎么覆盖整个张量，然后才决定具体算什么。

这是因为 kernel 的性能往往不是先被算法名决定，而是先被数据组织方式决定。块切得合不合理、映射是否连续、每个 program 的职责是否清晰，都会影响后面的访存和复用。

所以 Triton 看起来像在写“布局代码”，其实是在决定每个块如何靠近计算单元、如何减少无效搬运、以及如何把张量空间映射成更规整的执行单元。


### Q2 布局计算：Tile 形状带来多少边界填充

先考虑块怎么铺，再考虑块里怎么算。

In [ ]:
def tile_layout(rows, cols, block_rows, block_cols):
    """计算二维 Tile 网格、总 lane 数与边界填充比例。"""
    if min(rows, cols, block_rows, block_cols) <= 0:
        raise ValueError('shape values must be positive')
    row_programs = (rows + block_rows - 1) // block_rows
    col_programs = (cols + block_cols - 1) // block_cols
    useful_elements = rows * cols
    launched_elements = row_programs * col_programs * block_rows * block_cols
    return {
        'program_grid': (row_programs, col_programs),
        'useful_elements': useful_elements,
        'launched_elements': launched_elements,
        'padding_elements': launched_elements - useful_elements,
        'useful_ratio': useful_elements / launched_elements,
    }

for tile in [(16, 16), (32, 32), (64, 64)]:
    print(tile, '->', tile_layout(100, 130, *tile))
assert tile_layout(128, 128, 32, 32)['padding_elements'] == 0


## Q3：为什么 program_id、mask 和 tile 会在 Triton 里反复出现？


因为它们共同描述了“一个 program 该处理哪一块数据，以及越界时该怎么安全处理”。

- `program_id` 决定当前 program 的身份和位置；
- `tile` 决定它负责的数据块形状；
- `mask` 决定边界处哪些元素有效、哪些元素需要屏蔽。

这三者一起出现，说明 Triton 的核心不是单点计算，而是块级覆盖、边界处理和空间映射。如果你能在脑子里把这三者连起来，后面的 GEMM、softmax、FlashAttention kernel 会更容易读。

```mermaid
flowchart LR
    PID[program_id] --> TILE[tile]
    TILE --> MASK[mask]
    MASK --> SAFE[boundary-safe load/store]
    PID --> REGION[which region]
    REGION --> TILE
```


### Q3 边界验证：Program ID、偏移和 Mask 是否一致

先找 program_id、tile 和 mask，再看它怎么覆盖空间。

In [ ]:
def program_offsets(program_id, block_size, num_elements):
    """生成一个 Program 的偏移与边界 Mask，并验证有效索引不越界。"""
    if program_id < 0 or block_size <= 0 or num_elements <= 0:
        raise ValueError('invalid program configuration')
    offsets = [program_id * block_size + lane for lane in range(block_size)]
    mask = [offset < num_elements for offset in offsets]
    valid_offsets = [offset for offset, valid in zip(offsets, mask) if valid]
    return {
        'offsets': offsets,
        'mask': mask,
        'valid_offsets': valid_offsets,
        'valid_count': len(valid_offsets),
        'safe': all(offset < num_elements for offset in valid_offsets),
    }

tail = program_offsets(program_id=3, block_size=32, num_elements=100)
assert tail['valid_count'] == 4
assert tail['safe'] is True
assert tail['valid_offsets'] == [96, 97, 98, 99]
print(tail)


## ⚠️ 常见误区

- Triton 的 program 不是 Python 里的普通函数调用。
- block 不只是“一个小矩阵”，它更是一个 program 的责任范围。
- tile 不是为了好看，而是为了把空间映射和数据复用组织好。
- 先懂 block 思维，再看 kernel 代码，会比先背语法更有效。

## 相关阅读

**导语：** 把 Triton 的 block 模型继续接到入门 kernel、GEMM 和 FlashAttention，会更容易把空间划分和真实实现串起来。

- [01. Triton 入门与 Hello World：向量加法 (Vector Addition)](../03_Triton_Kernels/01_Triton_Vector_Addition.ipynb)
- [04. Triton 矩阵乘法 (GEMM) 与自动调优 (Autotune)](../03_Triton_Kernels/04_Triton_GEMM_Tutorial.ipynb)
- [08. Triton Flash Attention | 真正的 Flash Attention 前向算子](../03_Triton_Kernels/08_Triton_Flash_Attention.ipynb)
---